# Elementary flux modes and vectors (EFMs/EFVs)

Elementary flux modes (EFMs) are the smallest pathways a metabolic network can operate at steady state. StrainDesign does not enumerate EFMs by the double description method (as EFMtool does). Instead it can **find EFMs one by one with a MILP**, which works at genome scale: the shortest pathway producing a compound, all pathways up to a given length, or pathways that pass through a particular reaction. The trick is to treat *every* reaction as a knock-in candidate and ask for the cheapest set of reactions that can support a desired flux.

This chapter explains the formulation, shows worked examples on a small network and on *e_coli_core*, and explains how to recover the flux values of each computed pathway.

## Background

The steady-state fluxes of a network with stoichiometric matrix $\mathbf{S}$ form the **flux cone**
$$C = \{\mathbf{v} : \mathbf{S\,v = 0},\ v_i \ge 0 \ \text{for irreversible } i\}.$$
The *support* of a flux vector, $\mathrm{supp}(\mathbf{v}) = \{i : v_i \neq 0\}$, is the set of reactions it uses. An **EFM** is a nonzero $\mathbf{v}\in C$ with minimal support: no nonzero vector of $C$ uses a proper subset of its reactions. An EFM is determined by its support up to a positive scaling factor.

Most models also carry inhomogeneous constraints, such as a limited substrate uptake or a non-growth-associated ATP demand with a lower bound above zero. The flux space is then a **polyhedron** $P = \{\mathbf{v} : \mathbf{S\,v = 0},\ \mathbf{lb \le v \le ub}\}$ rather than a cone. **Elementary flux vectors** (EFVs) generalize EFMs to such polyhedra (Klamt et al., *PLoS Comput Biol* 13(4): e1005409, 2017).

## What StrainDesign computes

Given a region of interest $T$ described by linear inequalities (for instance $v_\text{growth}\ge 0.1$ or $v_\text{EtOH}\ge 1$), StrainDesign finds the **support-minimal flux vectors of $P\cap T$**. These are the reaction sets $D$ for which some $\mathbf{v}\in P\cap T$ uses only reactions in $D$, while no proper subset of $D$ would do.

* **In a flux cone with one reaction normalized** ($T = \{v_t \ge 1\}$, all bounds $0$ or $\pm\infty$), these sets are exactly the supports of the EFMs that use reaction $t$ in the forward direction. The correspondence is one-to-one, and the flux vector of each set is unique once $v_t$ is fixed.
* **With inhomogeneous bounds**, they are the minimal-support pathways of the polyhedron: the shortest EFVs that reach $T$. EFVs whose support contains that of another feasible vector are *not* returned, because the MILP searches for minimal supports. The flux values within a support need not be unique either (see below).

## The strain design formulation

| Ingredient | Setting | Meaning |
|---|---|---|
| intervention candidates | `ki_cost = {r.id: 1 for r in model.reactions}` | every reaction starts out **absent** and costs 1 to add |
| strain design module | `SDModule(model, PROTECT, constraints='v_t >= 1')` | the network that remains must be able to reach $T$ |
| objective | minimize the total intervention cost | fewest reactions, that is, minimal support |
| `solution_approach` | `BEST` with `max_solutions=1`, or `POPULATE` / `BEST` | one shortest pathway, or all of them in order of increasing length |
| `max_cost` | an integer | the largest support size to search |

The knock-in semantics carry the idea. A knock-in candidate carries no flux unless it is "added", and adding it restores its original bounds. Minimizing the number of additions under the PROTECT constraint therefore yields a minimal support. Once a design is found, StrainDesign excludes all its supersets and searches again. Enumeration thus produces the minimal supports as an antichain, smallest first. For how knock-ins are encoded in the MILP, see section 1.5 of the [Developer's Guide](../developers_guide.md).

Region constraints must exclude the zero vector, or the empty set would already be a solution. This is why a normalization such as `>= 1` is used rather than `> 0`.

## Prerequisites

In [1]:
import straindesign as sd
import cobra
import pandas as pd
from straindesign.names import *

solver = 'cplex'  # 'gurobi', 'scip' and 'glpk' work as well
cobra.Configuration().solver = solver

small = cobra.io.read_sbml_model('../../../tests/model_small_example.xml')
ecc = cobra.io.load_model('e_coli_core')

## Example 1: All EFMs that produce P in a small network

The small example network from the MCS chapter has 10 reactions. The metabolite P is exported through R3, and we ask for every elementary pathway that exports P. Every reaction is a knock-in candidate with cost 1, and the PROTECT module demands `R3 >= 1`.

In [2]:
for r in small.reactions:
    print(f'{r.id:4s} {r.build_reaction_string():14s} bounds={r.bounds}')

R1    --> A         bounds=(0.0, 100.0)
R2    <=> B         bounds=(-100.0, 100.0)
R3   P -->          bounds=(0.0, 100.0)
R4   E -->          bounds=(0.0, 100.0)
R5   A --> B        bounds=(0.0, 100.0)
R6   A --> C        bounds=(0.0, 100.0)
R7   A --> D        bounds=(0.0, 100.0)
R8   B <=> C        bounds=(-100.0, 100.0)
R9   B --> P        bounds=(0.0, 100.0)
R10  C + D --> E + P bounds=(0.0, 100.0)


In [3]:
efm_sols = sd.compute_strain_designs(small,
                                     sd_modules=[sd.SDModule(small, PROTECT, constraints='R3 >= 1')],
                                     ki_cost={r.id: 1 for r in small.reactions},
                                     solution_approach=POPULATE,
                                     solver=solver)
print(f'Status: {efm_sols.status}, {efm_sols.get_num_sols()} pathways found')
for design, cost in zip(efm_sols.get_reaction_sd(), efm_sols.sd_cost):
    print(f'{int(cost)} reactions: {sorted(design)}')

Status: optimal, 6 pathways found
3 reactions: ['R2', 'R3', 'R9']
4 reactions: ['R1', 'R3', 'R5', 'R9']
5 reactions: ['R1', 'R3', 'R6', 'R8', 'R9']
6 reactions: ['R1', 'R10', 'R3', 'R4', 'R6', 'R7']
7 reactions: ['R1', 'R10', 'R2', 'R3', 'R4', 'R7', 'R8']
7 reactions: ['R1', 'R10', 'R3', 'R4', 'R5', 'R7', 'R8']


Each design is a dictionary that maps the reactions that were "added" to `1.0`. The cost of a design equals the number of reactions it uses. The bounds of ±100 in this model cut off no pathway, so these six sets are exactly the six EFMs of the network's flux cone that export P. An independent check that enumerates reaction subsets with LPs gives the same six sets.

### Recovering the flux values

The design contains only the support. To get the flux vector, block every reaction outside the support and solve an LP. In a cone, the vector is unique up to scaling, so fixing the normalized reaction to 1 determines it completely:

In [4]:
def pathway_fluxes(model, support, constraints, obj=None, obj_sense='maximize'):
    # Flux vector that only uses the reactions in `support` and satisfies `constraints`
    with model as m:
        for r in m.reactions:
            if r.id not in support:
                r.bounds = (0.0, 0.0)
        sol = sd.fba(m, constraints=constraints, obj=obj, obj_sense=obj_sense, solver=solver)
    return sol.fluxes

efms = pd.DataFrame([pathway_fluxes(small, d, 'R3 = 1') for d in efm_sols.get_reaction_sd()]).fillna(0)
efms.index = [f'EFM {i+1}' for i in range(len(efms))]
efms.round(3)

,R1,R2,R3,R4,R5,R6,R7,R8,R9,R10
EFM 1,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
EFM 2,1.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0
EFM 3,1.0,0.0,1.0,0.0,0.0,1.0,0.0,-1.0,1.0,0.0
EFM 4,2.0,0.0,1.0,1.0,0.0,1.0,1.0,0.0,0.0,1.0
EFM 5,1.0,1.0,1.0,1.0,0.0,0.0,1.0,1.0,0.0,1.0
EFM 6,2.0,0.0,1.0,1.0,1.0,0.0,1.0,1.0,0.0,1.0


The sign of each entry gives the direction of a reversible reaction. R8, for example, runs B→C in some EFMs and C→B in others. The PROTECT constraint `R3 >= 1` fixes the direction of R3 only.

### All EFMs, not only those through one reaction

A single normalization row can cover every EFM that uses at least one irreversible reaction: require that the sum of all irreversible fluxes is at least 1. EFMs that consist only of reversible reactions are not covered (there are none here). For this we open the bounds to obtain the flux cone:

In [5]:
cone = small.copy()
for r in cone.reactions:
    r.bounds = (-1000 if r.lower_bound < 0 else 0, 1000)  # +-1000 is treated as unbounded
irreversible = [r.id for r in cone.reactions if r.lower_bound >= 0]
all_efms = sd.compute_strain_designs(cone,
                                     sd_modules=[sd.SDModule(cone, PROTECT, constraints=' + '.join(irreversible) + ' >= 1')],
                                     ki_cost={r.id: 1 for r in cone.reactions},
                                     solution_approach=BEST,
                                     solver=solver)
print(f'{all_efms.get_num_sols()} EFMs')
for d in all_efms.get_reaction_sd():
    print(sorted(d))

8 EFMs
['R1', 'R2', 'R5']
['R2', 'R3', 'R9']
['R1', 'R2', 'R6', 'R8']
['R1', 'R3', 'R5', 'R9']
['R1', 'R3', 'R6', 'R8', 'R9']
['R1', 'R10', 'R3', 'R4', 'R6', 'R7']
['R1', 'R10', 'R2', 'R3', 'R4', 'R7', 'R8']
['R1', 'R10', 'R3', 'R4', 'R5', 'R7', 'R8']


The eight sets agree with an exact nullspace rank test over all reaction subsets. Six of them export P, as found above, and the other two convert A to B, either directly (R5) or via C (R6, R8), and export B through R2.

## Example 2: EFVs in *e_coli_core*

*e_coli_core* has a glucose uptake limit of 10 mmol/gDW/h and a non-growth-associated ATP maintenance demand with $v_\text{ATPM}\ge 8.39$. Its flux space is a polyhedron, so the computed sets are EFVs rather than EFMs. Any reaction whose lower bound excludes zero must appear in every design. Here that is ATPM.

### a) The shortest pathway that supports growth

`BEST` with `max_solutions=1` returns one pathway of minimal size. The size is not known in advance, so `max_cost` is left open.

In [6]:
BM = 'BIOMASS_Ecoli_core_w_GAM'
ki_all = {r.id: 1 for r in ecc.reactions}

shortest = sd.compute_strain_designs(ecc,
                                     sd_modules=[sd.SDModule(ecc, PROTECT, constraints=f'{BM} >= 0.1')],
                                     ki_cost=ki_all,
                                     solution_approach=BEST,
                                     max_solutions=1,
                                     solver=solver)
growth_efv = sorted(shortest.get_reaction_sd()[0])
print(f'{len(growth_efv)} of {len(ecc.reactions)} reactions: {growth_efv}')

42 of 95 reactions: ['ACONTa', 'ACONTb', 'ADK1', 'ATPM', 'ATPS4r', 'BIOMASS_Ecoli_core_w_GAM', 'CO2t', 'CS', 'CYTBD', 'ENO', 'EX_co2_e', 'EX_glc__D_e', 'EX_h2o_e', 'EX_h_e', 'EX_nh4_e', 'EX_o2_e', 'EX_pi_e', 'G6PDH2r', 'GAPD', 'GLCpts', 'GLNS', 'GLUSy', 'GND', 'H2Ot', 'ICDHyr', 'NADH16', 'NADTRHD', 'NH4t', 'O2t', 'PDH', 'PGI', 'PGK', 'PGL', 'PGM', 'PIt2r', 'PPC', 'PPS', 'RPE', 'RPI', 'TALA', 'TKT1', 'TKT2']


Unlike in a cone, the flux values within a support are generally **not unique** in a polyhedron. An FVA restricted to the support shows the range that remains:

In [7]:
with ecc as m:
    for r in m.reactions:
        if r.id not in growth_efv:
            r.bounds = (0.0, 0.0)
    ranges = sd.fva(m, constraints=f'{BM} >= 0.1', solver=solver)
ranges.loc[[BM, 'EX_glc__D_e', 'EX_o2_e', 'ATPM']].round(3)

,minimum,maximum
BIOMASS_Ecoli_core_w_GAM,0.262,0.704
EX_glc__D_e,-10.000,-3.720
EX_o2_e,-29.225,-10.873
ATPM,8.390,22.552


A representative flux vector, for example the one that maximizes growth, can be obtained with `pathway_fluxes(ecc, growth_efv, f'{BM} >= 0.1', obj=BM)`.

### b) All growth-supporting pathways up to a given size

`POPULATE` enumerates pathways in order of increasing size. It uses CPLEX's and Gurobi's solution pools and is identical to `BEST` with SCIP and GLPK. Set `max_cost` to cap the size. The shortest pathways have 42 reactions, so `max_cost=43` returns them together with those one reaction longer:

In [8]:
from collections import Counter
growth_efvs = sd.compute_strain_designs(ecc,
                                        sd_modules=[sd.SDModule(ecc, PROTECT, constraints=f'{BM} >= 0.1')],
                                        ki_cost=ki_all,
                                        solution_approach=POPULATE,
                                        max_cost=43,
                                        solver=solver)
print(f'Status: {growth_efvs.status}')
print('Pathways per size:', dict(sorted(Counter(len(d) for d in growth_efvs.get_reaction_sd()).items())))

Status: optimal
Pathways per size: {42: 4, 43: 12}


The number of pathways grows quickly with `max_cost`. We measured 16 pathways up to size 43, 88 up to 44 and 320 up to 45 (CPLEX, 4 threads, 28 s, 60 s and 100 s). For larger bounds, limit the search with `max_solutions` or `time_limit`. The status reports whether the enumeration finished: `optimal` means it did, while `time_limit_w_sols` means the result is incomplete.

### c) Pathways through a given reaction

To restrict the search to pathways that use a particular reaction in a particular direction, put that reaction into the PROTECT constraint. Here we enumerate the ethanol-secreting pathways of up to 22 reactions:

In [9]:
etoh_efvs = sd.compute_strain_designs(ecc,
                                      sd_modules=[sd.SDModule(ecc, PROTECT, constraints='EX_etoh_e >= 1')],
                                      ki_cost=ki_all,
                                      solution_approach=POPULATE,
                                      max_cost=22,
                                      solver=solver)
for d in sorted(etoh_efvs.get_reaction_sd(), key=len):
    print(len(d), sorted(d))

20 ['ACALD', 'ALCD2x', 'ATPM', 'ATPS4r', 'CO2t', 'ENO', 'ETOHt2r', 'EX_co2_e', 'EX_etoh_e', 'EX_glc__D_e', 'FBA', 'GAPD', 'GLCpts', 'PDH', 'PFK', 'PGI', 'PGK', 'PGM', 'PYK', 'TPI']
21 ['ACALD', 'ALCD2x', 'ATPM', 'CO2t', 'ENO', 'ETOHt2r', 'EX_co2_e', 'EX_etoh_e', 'EX_glc__D_e', 'FBA', 'FORt', 'FORt2', 'GAPD', 'GLCpts', 'PDH', 'PFK', 'PGI', 'PGK', 'PGM', 'PYK', 'TPI']
21 ['ACALD', 'ALCD2x', 'ATPM', 'CO2t', 'ENO', 'ETOHt2r', 'EX_co2_e', 'EX_etoh_e', 'EX_glc__D_e', 'FBA', 'GAPD', 'GLCpts', 'PDH', 'PFK', 'PGI', 'PGK', 'PGM', 'PYK', 'SUCCt2_2', 'SUCCt3', 'TPI']
21 ['ACALD', 'ALCD2x', 'ATPM', 'CO2t', 'ENO', 'ETOHt2r', 'EX_co2_e', 'EX_etoh_e', 'EX_glc__D_e', 'FBA', 'GAPD', 'GLCpts', 'NADTRHD', 'PDH', 'PFK', 'PGI', 'PGK', 'PGM', 'PYK', 'THD2', 'TPI']
22 ['ACALD', 'ALCD2x', 'ATPM', 'ATPS4r', 'CO2t', 'ENO', 'ETOHt2r', 'EX_co2_e', 'EX_etoh_e', 'EX_glc__D_e', 'FBA', 'GAPD', 'GLCpts', 'MDH', 'ME1', 'PDH', 'PFK', 'PGI', 'PGK', 'PGM', 'PPC', 'TPI']
22 ['ACALD', 'ALCD2x', 'ATPM', 'ENO', 'ETOHt2r', 'EX_

Further constraints narrow the search in the same way. For example, `constraints=['EX_etoh_e >= 1', 'EX_o2_e = 0']` restricts it to anaerobic pathways. Put all conditions into one PROTECT module. With two PROTECT modules, each region must be reachable on its own, possibly by different flux vectors, so the resulting sets are no longer supports of single pathways.

### d) EFMs of the flux cone

To obtain EFMs instead of EFVs, replace all bounds by $0$ or $\pm\infty$ first, which removes the ATP maintenance demand and the uptake limit. With the normalization `EX_etoh_e = 1`, the flux vector of each EFM is unique:

In [10]:
ecc_cone = ecc.copy()
for r in ecc_cone.reactions:
    r.bounds = (-1000 if r.lower_bound < 0 else 0, 1000)
etoh_efms = sd.compute_strain_designs(ecc_cone,
                                      sd_modules=[sd.SDModule(ecc_cone, PROTECT, constraints='EX_etoh_e >= 1')],
                                      ki_cost={r.id: 1 for r in ecc_cone.reactions},
                                      solution_approach=POPULATE,
                                      max_cost=20,
                                      solver=solver)
flux = pd.DataFrame([pathway_fluxes(ecc_cone, d, 'EX_etoh_e = 1') for d in etoh_efms.get_reaction_sd()]).fillna(0)
flux.loc[:, (flux.abs() > 1e-9).any()].T.round(3)

,0,1
PFK,0.50,1.75
PGI,0.50,0.50
PGK,-1.00,-1.00
ACALD,-1.00,-1.00
PGM,-1.00,-1.00
ALCD2x,-1.00,-1.00
ATPM,1.25,0.00
ATPS4r,0.25,0.25
PYK,0.50,0.50
CO2t,-1.00,-1.00


## Caveats and practical notes

* **Enumeration grows combinatorially.** Genome-scale networks have millions to billions of EFMs. Use `max_cost` as a cap on the pathway length, and `max_solutions` and `time_limit` as budgets. The first pathway is usually quick to find, but proving that none remain up to `max_cost` can take much longer.
* **Costs count reactions of the original model.** StrainDesign compresses the network before building the MILP. A lump of coupled reactions costs the sum of its members, and a lump of parallel reactions is expanded back into alternative designs. `sd_cost` and `max_cost` therefore always refer to the number of original reactions. Designs are returned with original reaction IDs. Compression is lossless (`compress=False` returns the same sets) and should stay on: without it, enumeration b) had found only 2 of the 16 pathways when it hit a 20-minute time limit. Since one compressed MILP solution can expand into several designs, `max_solutions` counts MILP solutions, not returned pathways.
* **Use `BEST` or `POPULATE`, not `ANY`.** `ANY` finds valid designs quickly, but in our tests on the small test models it also returned sets that are *not* support-minimal (a subset of reactions would have sufficed). If you use `ANY`, verify minimality with an LP: removing any single reaction must make the target region infeasible.
* **Solver notes.** CPLEX and Gurobi run `POPULATE` natively. With GLPK and SCIP, StrainDesign emulates it with repeated solves, and `BEST` gives the same result faster. GLPK has no indicator constraints and uses big-M constraints instead (M=1000 by default). This is only exact if no flux can exceed M. Use a fast MILP solver for genome-scale models.
* **Essential reactions are forced in.** A reaction without which the region is infeasible (for example ATPM, whose lower bound is above zero, or the glucose uptake when glucose is the only carbon source) appears in every pathway. StrainDesign detects such reactions in advance and fixes them.
* **Non-unique flux values.** A design is a support. In a cone with one normalized reaction the flux vector is unique. With inhomogeneous bounds, a whole polytope of flux vectors may share the support. Use FBA or FVA restricted to the support, as shown above, to select or characterize them.
* **Region definition.** The PROTECT constraints must exclude $\mathbf{v}=0$. With a reversible target reaction, `v_t >= 1` and `v_t <= -1` select the two directions separately.